In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

In [2]:
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [4]:
store = InMemoryStore()
checkpointer = InMemorySaver()

In [5]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=api_key
)

In [6]:
class TicketRequest(BaseModel):
    tickets: int = Field(
        description="Number of movie tickets to book"
    )
    movie_name: str = Field(
        description="Name of the movie"
    )


@tool
def book_ticket(
    request_details: TicketRequest,
    runtime: ToolRuntime
):
    """
    Book movie tickets for the user.
    """

    user_name = runtime.context.user_name

    # Read existing bookings
    item = runtime.store.get(("user-ticket-map",), user_name)
    tickets = item.value.get("bookings", []) if item else []


    # Pydantic has already validated these
    number_of_tickets = request_details.tickets
    movie_name = request_details.movie_name

    if number_of_tickets <= 0 or number_of_tickets > 10:
        return "Please book between 1 and 10 tickets only."

    # Generate ticket IDs
    ticket_ids = [
        str(uuid.uuid4())
        for _ in range(number_of_tickets)
    ]

    ticket_obj = {
        "total_tickets": number_of_tickets,
        "ticket_ids": ticket_ids,
        "movie_name": movie_name,
    }

    tickets.append(ticket_obj)

    # Persist
    runtime.store.put(
        ("user-ticket-map",),
        user_name,
        {"bookings": tickets}
    )

    return (
        f"Hello {user_name}, {number_of_tickets} tickets "
        f"have been booked for {movie_name}. "
        f"Ticket IDs: {ticket_ids}"
    )

In [7]:
@tool
def get_ticket_details(runtime: ToolRuntime):
    """
    getting the list of booked tickets by the user
    """

    user_name = runtime.context.user_name
    ticket_item = runtime.store.get(("user-ticket-map",), user_name)
    ticket_list = ticket_item.value.get("bookings", []) if ticket_item else []

    if len(ticket_list) == 0:
        return "no booking has been made yet"
    else:
        return ticket_list


In [8]:
@tool
def cancel_booking(ticket_id: str, runtime: ToolRuntime):
    """
    Cancel a movie ticket using its ticket ID.
    The ticket can only be cancelled by the user who owns it.
    """

    user_name = runtime.context.user_name

    # 1. Get user's existing bookings
    item = runtime.store.get(("user-ticket-map",), user_name)

    if item is None:
        return f"No bookings found for {user_name}."

    ticket_list = item.value.get("bookings", [])

    if not ticket_list:
        return f"No bookings found for {user_name}."

    # 2. Search for the ticket ID
    for booking in ticket_list:

        if ticket_id in booking["ticket_ids"]:

            # Save information for response
            movie_name = booking["movie_name"]

            # 3. Remove the ticket ID
            booking["ticket_ids"].remove(ticket_id)

            # 4. Decrease ticket count
            booking["total_tickets"] -= 1

            # 5. If no tickets remain, remove the booking
            if booking["total_tickets"] == 0:
                ticket_list.remove(booking)

            # 6. Persist updated bookings
            runtime.store.put(
                ("user-ticket-map",),
                user_name,
                {"bookings": ticket_list}
            )

            return (
                f"Ticket {ticket_id} has been cancelled successfully "
                f"for movie '{movie_name}'."
            )

    # 7. Ticket wasn't found
    return (
        f"Ticket ID '{ticket_id}' was not found in your bookings."
    )

In [9]:
@dataclass
class Context:
    user_name:str
    email:str

In [10]:
agent = create_agent(
    model=llm,
    tools=[book_ticket, get_ticket_details, cancel_booking],
    store=store,
    context_schema=Context,
    checkpointer=checkpointer,
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "cancel_booking": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "get_ticket_details": False,
                "book_ticket": False
            }
        ),
    ]
)

In [11]:
context = Context(
    user_name="shradha",
    email="shradha@gmail.com"
)

In [12]:
config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

In [14]:
response_v1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "book 2 tickets for stree"}
        ]
    },
    config=config,
    context=context
)

# print(response_v1)
print(response_v1["messages"][-1].content)

c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


2 tickets have been successfully booked for the movie "Stree." If you need any further assistance, feel free to 
ask!

In [15]:
response_v2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Get the ticket details with ticket id"}
        ]
    },
    config=config,
    context=context
)

# print(response_v2)
print(response_v2["messages"][-1].content)

c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


You have 2 tickets booked for the movie "Stree." The ticket IDs are:
1. adab54a8-2142-483d-b15c-50d4baec5da2
2. bb0edc0e-7219-4b77-8dbb-34668860e17b

If you need any more information or assistance, please let me know!

In [16]:
response_three = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Cancel the ticket with ticket id bb0edc0e-7219-4b77-8dbb-34668860e17b"}
        ]
    },
    config=config,
    context=context
)

#print(response_three)
print(response_three["messages"][-1].content)


In [17]:
# need to pass the context if the cancellation tool is expecting it.
response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    context=context
)
print(response)
print(response["messages"][-1].content)

c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Asus\Desktop\agentic ai\ai-ml\.venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='shradh...ail='shradha@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{
    'messages': [
        HumanMessage(
            content='book 2 tickets for stree',
            additional_kwargs={},
            response_metadata={},
            id='e17589eb-91fa-4fb3-89c2-1286404b7908'
        ),
        HumanMessage(
            content='book 2 tickets for stree',
            additional_kwargs={},
            response_metadata={},
            id='bb819843-e91d-4287-a3cd-b23541411671'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 141,
                    'total_tokens': 164,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-4.1-mini-2025-04-14',
                'system_fingerprint': 'fp_2738b29278',
                'id': 'chatcmpl-EeeHZWY32DTZMBak9ydm74QUnORZU',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a18c63-0c12-753d-a2b8-49e18e7d19dc-0',
            tool_calls=[
                {
                    'name': 'book_ticket',
                    'args': {'request_details': {'tickets': 2, 'movie_name': 'stree'}},
                    'id': 'call_aDpDglWr219xYMl2n9c40tZa',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 141,
                'output_tokens': 23,
                'total_tokens': 164,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Hello shradha, 2 tickets have been booked for stree. Ticket IDs: 
['adab54a8-2142-483d-b15c-50d4baec5da2', 'bb0edc0e-7219-4b77-8dbb-34668860e17b']",
            name='book_ticket',
            id='a55566d1-88ed-4b8a-901b-1d5da44ca34f',
            tool_call_id='call_aDpDglWr219xYMl2n9c40tZa'
        ),
        AIMessage(
            content='2 tickets have been successfully booked for the movie "Stree." If you need any further 
assistance, feel free to ask!',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 26,
                    'prompt_tokens': 241,
                    'total_tokens': 267,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-4.1-mini-2025-04-14',
                'system_fingerprint': 'fp_2738b29278',
                'id': 'chatcmpl-EeeHbyd

The ticket with ID bb0edc0e-7219-4b77-8dbb-34668860e17b has been successfully cancelled for the movie "Stree." If 
you need any further assistance, please let me know!